In [2]:
import tenseal as ts
import base64
import os
CLIENT_FILES = [
    "client1/enc_model_client1.txt",
    "client2/enc_model_client2.txt"
]
CONTEXT_FILE = "context/key_public.txt"
OUTPUT_FILE = "server/enc_avg.txt"

# Charger le contexte public
with open(CONTEXT_FILE, "r") as f:
    context = ts.context_from(base64.b64decode(f.read()))

def load_encrypted_model(path):
    model = {}
    with open(path, "r") as f:
        lines = [line.strip() for line in f if line.strip()]
    for i in range(0, len(lines), 2):
        key = lines[i]
        vec = ts.ckks_vector_from(context, base64.b64decode(lines[i + 1]))
        model[key] = vec
    return model

def average_models(models):
    avg = {}
    for key in models[0]:
        avg[key] = sum(m[key] for m in models) * (1 / len(models))
    return avg

models = [load_encrypted_model(file) for file in CLIENT_FILES]
avg_model = average_models(models)

os.makedirs(os.path.dirname(OUTPUT_FILE), exist_ok=True)

with open(OUTPUT_FILE, "w") as f:
    for key, vec in avg_model.items():
        f.write(f"{key}\n{base64.b64encode(vec.serialize()).decode()}\n\n")
print("✅ Moyenne homomorphique exportée en Base64 dans", OUTPUT_FILE)


✅ Moyenne homomorphique exportée en Base64 dans server/enc_avg.txt
